In [ ]:
!pip install openpyxl

In [7]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

26/01/20 10:31:51 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 138090 ms exceeds timeout 120000 ms
26/01/20 10:31:51 WARN SparkContext: Killing executors is not supported by current scheduler.


In [20]:
## rds connection credentials
jdbc_url = "jdbc:postgresql://my-first-rds.cniuuy0iu2av.eu-north-1.rds.amazonaws.com:5432/testdb"

jdbc_props = {
    "user": "postgres",
    "password": "admin123",
    "driver": "org.postgresql.Driver"
}


In [ ]:
members_df = read_excel("s3://titan-glue-test-data/input_data/member_data/MemberDetailedProfile_till_oct_24_2025.xlsx")
transactions_df = read_excel("s3://titan-glue-test-data/input_data/transaction_data/Detailed_Transaction_till_Oct_25_2025.xlsx")
campaigns_df = read_excel("s3://titan-glue-test-data/input_data/campaign_data/CampaignData_Apr_to_Sep.xlsx")

- first we convert the excel files to csv file and store back to s3

In [14]:
import boto3
import io
import csv
from openpyxl import load_workbook

def convert_excel_s3_to_csv_s3(
    input_bucket: str,
    input_key: str,
    output_bucket: str,
    output_key: str,
    sheet_name: str = None
):
    """
    Stream an Excel file from S3, convert to CSV row-by-row, and write back to S3.
    Safe for large Excel files. No Spark. No local disk.
    """

    s3 = boto3.client("s3")
    # Read Excel from S3 as binary stream
    obj = s3.get_object(Bucket=input_bucket, Key=input_key)
    excel_stream = io.BytesIO(obj["Body"].read())

    # Load workbook in streaming (read-only) mode
    wb = load_workbook(
        excel_stream,
        read_only=True,
        data_only=True
    )

    ws = wb[sheet_name] if sheet_name else wb.active

    # Stream rows into CSV buffer
    csv_buffer = io.StringIO()
    writer = csv.writer(csv_buffer)

    for row in ws.iter_rows(values_only=True):
        writer.writerow(row)

    wb.close()

    # Upload CSV back to S3
    s3.put_object(
        Bucket=output_bucket,
        Key=output_key,
        Body=csv_buffer.getvalue().encode("utf-8")
    )

    print(f"✅ Converted s3://{input_bucket}/{input_key} → s3://{output_bucket}/{output_key}")


In [15]:
# member
convert_excel_s3_to_csv_s3(
    input_bucket="titan-glue-test-data",
    input_key="input_data/member_data/MemberDetailedProfile_till_oct_24_2025.xlsx",
    output_bucket="titan-glue-test-data",
    output_key="input_csv/member_data/member_data.csv"
)

# transaction
convert_excel_s3_to_csv_s3(
    input_bucket="titan-glue-test-data",
    input_key="input_data/transaction_data/Detailed_Transaction_till_Oct_25_2025.xlsx",
    output_bucket="titan-glue-test-data",
    output_key="input_csv/transaction_data/transaction_data.csv"
)

# campaign
convert_excel_s3_to_csv_s3(
    input_bucket="titan-glue-test-data",
    input_key="input_data/campaign_data/CampaignData_Apr_to_Sep.xlsx",
    output_bucket="titan-glue-test-data",
    output_key="input_csv/campaign_data/campaign_data.csv"
)

✅ Converted s3://titan-glue-test-data/input_data/member_data/MemberDetailedProfile_till_oct_24_2025.xlsx → s3://titan-glue-test-data/input_csv/member_data/member_data.csv
✅ Converted s3://titan-glue-test-data/input_data/transaction_data/Detailed_Transaction_till_Oct_25_2025.xlsx → s3://titan-glue-test-data/input_csv/transaction_data/transaction_data.csv
✅ Converted s3://titan-glue-test-data/input_data/campaign_data/CampaignData_Apr_to_Sep.xlsx → s3://titan-glue-test-data/input_csv/campaign_data/campaign_data.csv


* now, write these three csv to rds and work on them to create consolidated and other tables
* Use overwrite only once (initial load).
* From the next run onward, always use append + dedup / watermark logic.

In [ ]:
## initial digestion
#member_data
df_members = spark.read \
    .option("header", "true") \
    .csv("s3://titan-glue-test-data/input_csv/member_data/member_data.csv")

df_members.write \
    .mode("overwrite") \
    .jdbc(jdbc_url, "membersdata", properties=jdbc_props)
print('Completed')

In [ ]:
## initial digestion
#transaction data
df_transaction = spark.read \
    .option("header", "true") \
    .csv("s3://titan-glue-test-data/input_csv/transaction_data/transaction_data.csv")

df_transaction.write \
    .mode("overwrite") \
    .jdbc(jdbc_url, "transactionssdata", properties=jdbc_props)
print('Completed')

In [ ]:
## initial digestion
#campaign data
df_campaign = spark.read \
    .option("header", "true") \
    .csv("s3://titan-glue-test-data/input_csv/campaign_data/campaign_data.csv")

df_campaign.write \
    .mode("overwrite") \
    .jdbc(jdbc_url, "campaignssdata", properties=jdbc_props)
print('Completed')

26/01/20 12:40:53 INFO BlockManagerInfo: Removed broadcast_8_piece0 on 880bc9e39da9:36529 in memory (size: 52.2 KiB, free: 366.1 MiB)
26/01/20 12:40:53 INFO BlockManagerInfo: Removed broadcast_18_piece0 on 880bc9e39da9:36529 in memory (size: 52.2 KiB, free: 366.2 MiB)
26/01/20 12:40:53 INFO BlockManagerInfo: Removed broadcast_19_piece0 on 880bc9e39da9:36529 in memory (size: 14.6 KiB, free: 366.2 MiB)
26/01/20 12:40:53 INFO BlockManagerInfo: Removed broadcast_9_piece0 on 880bc9e39da9:36529 in memory (size: 25.5 KiB, free: 366.2 MiB)
26/01/20 12:40:53 INFO BlockManagerInfo: Removed broadcast_13_piece0 on 880bc9e39da9:36529 in memory (size: 52.2 KiB, free: 366.3 MiB)
26/01/20 12:40:53 INFO BlockManagerInfo: Removed broadcast_14_piece0 on 880bc9e39da9:36529 in memory (size: 14.7 KiB, free: 366.3 MiB)
